# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmRaj6666/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

My Rule

I assign a higher baseline score to pages that show multiple observed signals of needing a content refresh. The rule increases the score when a page has a declining traffic trend, a low click-through rate (CTR), a reasonable average search position, and low engagement. The output is a ranked review queue that helps prioritize pages for manual review. This is a decision-support rule rather than a prediction model.

Reason Codes
| Reason Code     | Meaning                                               |
| --------------- | ----------------------------------------------------- |
| LOW_CTR         | Click-through rate is lower than expected.            |
| DECLINING_TREND | Search trend is decreasing.                           |
| LOW_ENGAGEMENT  | Engagement rate is low.                               |
| REVIEW_PRIORITY | Multiple signals suggest the page should be reviewed. |


In [12]:
!git clone https://github.com/OmRaj6666/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 138 (delta 50), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 2.24 MiB | 8.33 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [13]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship


In [14]:
!find data -name "*.csv"

data/raw/content_refresh_anonymized.csv


In [15]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [16]:
!pwd
!ls

/content/flyrank-ml-internship/flyrank-ml-internship
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
import pandas as pd
import os

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Build baseline score
df["baseline_score"] = 0

df.loc[df["trend_direction"] == "down", "baseline_score"] += 40
df.loc[df["ctr"] < 0.10, "baseline_score"] += 30
df.loc[df["avg_position"] < 20, "baseline_score"] += 20
df.loc[df["engagement_rate"] < 2, "baseline_score"] += 10

# Default reason
df["reason_code"] = "REVIEW_PRIORITY"

df.loc[
    (df["trend_direction"] == "down") & (df["ctr"] >= 0.10),
    "reason_code"
] = "DECLINING_TREND"

df.loc[
    (df["ctr"] < 0.10),
    "reason_code"
] = "LOW_CTR"

df.loc[
    (df["engagement_rate"] < 2) &
    (df["ctr"] >= 0.10) &
    (df["trend_direction"] != "down"),
    "reason_code"
] = "LOW_ENGAGEMENT"

# More specific reason
df.loc[df["trend_direction"] == "down", "reason_code"] = "DECLINING_TREND"
df.loc[df["ctr"] < 0.10, "reason_code"] = "LOW_CTR"
df.loc[df["engagement_rate"] < 2, "reason_code"] = "LOW_ENGAGEMENT"

df["action"] = "Refresh Content"

# Rank pages
df = df.sort_values("baseline_score", ascending=False)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)
df.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(df[["content_id","baseline_score","reason_code","action"]].head(20))
print("\nCSV saved successfully.")

                 content_id  baseline_score     reason_code           action
25     content_033ae3e7aecf             100  LOW_ENGAGEMENT  Refresh Content
29977  content_c87291853cab             100  LOW_ENGAGEMENT  Refresh Content
22884  content_7107cf349e26             100  LOW_ENGAGEMENT  Refresh Content
46     content_c59d46264834             100  LOW_ENGAGEMENT  Refresh Content
52     content_a4cd54c0a5f1             100  LOW_ENGAGEMENT  Refresh Content
54     content_ff8ea1364b59             100  LOW_ENGAGEMENT  Refresh Content
58     content_caff51984338             100  LOW_ENGAGEMENT  Refresh Content
29952  content_e6cc2aad65ea             100  LOW_ENGAGEMENT  Refresh Content
22889  content_68047ee64a1e             100  LOW_ENGAGEMENT  Refresh Content
22892  content_b8366b9ef4e8             100  LOW_ENGAGEMENT  Refresh Content
22905  content_871887faeb84             100  LOW_ENGAGEMENT  Refresh Content
22907  content_f9563540cd75             100  LOW_ENGAGEMENT  Refresh Content

| Rank  | Action          | Reason Code     | Confidence | What would make it wrong              |
| ----- | --------------- | --------------- | ---------- | ------------------------------------- |
| 1     | Refresh Content | DECLINING_TREND | High       | Traffic decline may be seasonal.      |
| 2     | Refresh Content | LOW_CTR         | High       | Title may have been recently updated. |
| 3     | Refresh Content | LOW_ENGAGEMENT  | Medium     | Engagement data may be incomplete.    |
| 4     | Refresh Content | LOW_CTR         | Medium     | Search intent may have changed.       |
| 5     | Refresh Content | DECLINING_TREND | High       | Recent updates not reflected yet.     |
| 6     | Refresh Content | LOW_ENGAGEMENT  | Medium     | Temporary traffic fluctuations.       |
| 7     | Refresh Content | LOW_CTR         | Medium     | Metadata recently improved.           |
| 8     | Refresh Content | DECLINING_TREND | High       | Seasonal variation.                   |
| 9     | Refresh Content | LOW_CTR         | Medium     | Data collection delay.                |
| 10    | Refresh Content | LOW_ENGAGEMENT  | Medium     | Tracking issue.                       |
| 11–20 | Refresh Content | Mixed Signals   | Medium     | Require manual review before action.  |


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Some pages may receive high scores because of temporary traffic drops, seasonal effects, or incomplete engagement data rather than genuine content quality problems. These cases should be manually reviewed before taking action.

This baseline rule only uses observed historical search and engagement signals that were available at the decision time. It does not use future information, product flags, or label-derived fields, helping avoid data leakage.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.